# Baseline parameter grid

This notebook demonstrates how to evaluate different pipeline settings for individual identification. The helper `run_simple_baseline_all_species` is executed for each parameter combination.

In [ ]:
from pathlib import Path
from itertools import product
import pandas as pd
from sklearn.model_selection import ParameterGrid

from FIT_python.pipeline_individual_id.simple_baseline import run_simple_baseline_all_species, collect_id_metrics

EXP_ROOT = Path('results/grid_baseline')
EXP_ROOT.mkdir(parents=True, exist_ok=True)

param_grid = {
    'k_features': [10],
    'selection_method': ['forward'],
    'reducers': ['lda', 'umap'],
    'use_sexmodel_prediction': [True, False],
    'scaler_methods': ['standard'],
}

grids = list(ParameterGrid(param_grid))
print(f"Total runs: {len(grids)}")


In [ ]:
all_results = []

for i, params in enumerate(grids, start=1):
    run_dir = EXP_ROOT / f"run_{i}"
    run_simple_baseline_all_species(
        run_dir,
        best_k=params['k_features'],
        cutoff={},
        selection_method=params['selection_method'],
        reducers=params['reducers'],
        scaler_methods=params['scaler_methods'],
        use_sexmodel_prediction=params['use_sexmodel_prediction'],
        reuse_summary=True,
    )

    df = collect_id_metrics(run_dir)
    df['run_id'] = i
    for key, value in params.items():
        df[key] = value
    all_results.append(df)

results_df = pd.concat(all_results, ignore_index=True)
results_fp = EXP_ROOT / 'summary_grid.csv'
results_df.to_csv(results_fp, index=False)
results_df.head()


After running the grid search you can combine the per-species fold results:
```python
from FIT_python.utils.aggregate_all_folds import aggregate_all_folds
aggregate_all_folds(EXP_ROOT / 'fold_cv')
```
